<a href="https://colab.research.google.com/github/junseo25/ds2002-walmart-hurricane-project/blob/main/Weather_cleaning_for_Walmart_Midterm_scratch_work.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#copy past from previous labs and studios | From 2026-10-07 — APIs 101 and Ingestion with Retries — Studio


In [ ]:
import requests, pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta


BASE = 'https://archive-api.open-meteo.com/v1/archive'

END = date.today() - timedelta(days=3)     # let the archive settle
START = END - timedelta(days=2)            # a three-day window

params = {
    'latitude': 38.03, 'longitude': -78.51,          # Charlottesville
    'start_date': START.isoformat(),
    'end_date': END.isoformat(),
    'hourly': 'temperature_2m,precipitation',
    'timezone': 'America/New_York',
}

print(BASE + '?' + urlencode(params))



resp = requests.get(BASE, params=params, timeout=15)

print('status:', resp.status_code)
print('keys:', list(resp.json().keys()))
print('units:', resp.json()['hourly_units'])


hourly = pd.DataFrame(resp.json()['hourly'])
hourly['time'] = pd.to_datetime(hourly['time'])
print(hourly.shape)
hourly.head()


wettest = hourly.loc[hourly['precipitation'].idxmax()]
print('wettest hour:', wettest['time'], '|', wettest['precipitation'], 'mm')

https://archive-api.open-meteo.com/v1/archive?latitude=38.03&longitude=-78.51&start_date=2026-10-27&end_date=2026-10-29&hourly=temperature_2m%2Cprecipitation&timezone=America%2FNew_York
status: 200
keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
units: {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm'}
(72, 3)
wettest hour: 2026-10-27 00:00:00 | 0.0 mm


#Need this to be editable and malleable for other time zones and lat long coordinates based on data file


In [ ]:
from pandas.io.api import read_csv
# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")


In [ ]:
import requests
import pandas as pd
from urllib.parse import urlencode
from datetime import date, timedelta

#Since every store is in the same state, just set all state rows to the same value.
store['state'] = 'FL'

#Tallahassee is missing coords instead of dropping a store location
store.loc[store['city'] == 'Tallahassee', ['latitude', 'longitude']] = [30.4383, -84.2807]

#drop any duplicate rows that may have the same latitude and longitude
store = store.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)

#copied from lab
END = date.today() - timedelta(days=3)
START = END - timedelta(days=2)

BASE = 'https://archive-api.open-meteo.com/v1/archive'

# 4. Iterate through each store and query the API
all_weather_data = []

for i in range(len(store)):
    store_id = store.loc[i, 'store_id']
    city = store.loc[i, 'city']
    lat = store.loc[i, 'latitude']
    lon = store.loc[i, 'longitude']
    state = store.loc[i, 'state']

    # Assign appropriate timezone (Pensacola is in the Central Time Zone)
    tz = 'America/Chicago' if city == 'Pensacola' else 'America/New_York'

    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': START.isoformat(),
        'end_date': END.isoformat(),
        'hourly': 'temperature_2m,precipitation',
        'timezone': tz,
    }

    response = requests.get(BASE, params=params)

    if response.status_code == 200:
        data = response.json()
        print(f"Successfully fetched data for {store_id} ({city}, {state})")

        hourly = data.get('hourly', {})
        if hourly:
            temp_stores = pd.DataFrame({
                'store_id': store_id,
                'city': city,
                'state': state,
                'time': hourly.get('time'),
                'temperature_2m': hourly.get('temperature_2m'),
                'precipitation': hourly.get('precipitation')
            })
            all_weather_data.append(temp_stores)
    else:
        print(f"Failed to fetch data for {store_id} ({city}): Status {response.status_code}")

# Combine into a master DataFrame
if all_weather_data:
    master_weather_df = pd.concat(all_weather_data, ignore_index=True)
    print(master_weather_df.head())

Successfully fetched data for FL-239 (Orlando, FL)
Successfully fetched data for FL-105 (Tampa, FL)
Successfully fetched data for FL-330 (Miami, FL)
Successfully fetched data for FL-412 (Jacksonville, FL)
Successfully fetched data for FL-287 (Kissimmee, FL)
Successfully fetched data for FL-156 (Daytona Beach, FL)
Successfully fetched data for FL-501 (Fort Myers, FL)
Successfully fetched data for FL-078 (Gainesville, FL)
Successfully fetched data for FL-619 (Tallahassee, FL)
Successfully fetched data for FL-445 (St. Petersburg, FL)
Successfully fetched data for FL-302 (Lakeland, FL)
Successfully fetched data for FL-710 (Pensacola, FL)
  store_id     city state              time  temperature_2m  precipitation
0   FL-239  Orlando    FL  2026-10-27T00:00            19.8            0.0
1   FL-239  Orlando    FL  2026-10-27T01:00            19.2            0.0
2   FL-239  Orlando    FL  2026-10-27T02:00            19.1            0.0
3   FL-239  Orlando    FL  2026-10-27T03:00            19.